In [70]:
# Đọc và lọc dữ liệu đầu vào
import pandas as pd

data = pd.read_csv("mental_health.csv")
data.columns = data.columns.str.strip().str.lower()
data = data[data["gender"].isin(["Male", "Female"])].reset_index(drop=True)
data["employee_id"] = range(1, len(data) + 1)

print(f"Kích thước dữ liệu sau khi lọc: {data.shape}")
print(data["gender"].value_counts())
data.head()

Kích thước dữ liệu sau khi lọc: (95002, 22)
gender
Male      65243
Female    29759
Name: count, dtype: int64


,employee_id,age,gender,job_role,seniority_level,years_experience,work_mode,salary_usd,work_hours_per_week,sleep_hours_per_night,...,deadline_pressure_score,stress_score,burnout_score,phq9_score,phq9_category,gad7_score,gad7_category,burnout_level,seeks_mental_health_support,job_change_intention
0,1,33,Male,Software Engineer,Lead,11,Remote,40000,55,6.3,...,8.3,10.0,10.0,19,Moderately Severe (15-19),12,Moderate (10-14),Severe,1,1
1,2,22,Female,Data Analyst,Principal,0,Remote,57714,50,6.8,...,2.6,7.1,7.4,9,Mild (5-9),7,Mild (5-9),Severe,0,1
2,3,32,Male,Data Analyst,Junior,10,Hybrid,42823,41,3.6,...,4.3,6.7,3.8,2,None (0-4),5,Mild (5-9),Moderate,0,1
3,4,29,Female,DevOps Engineer,Mid,8,Hybrid,77421,39,6.7,...,10.0,7.6,3.9,6,Mild (5-9),8,Mild (5-9),Moderate,0,0
4,5,31,Male,Software Engineer,Mid,9,Hybrid,98067,45,6.9,...,5.4,3.2,2.2,0,None (0-4),0,Minimal (0-4),Low,0,0


In [71]:
# Tiền xử lý 1: Loại bỏ bản ghi trùng lặp
model_data = data.copy()
model_data = model_data.drop_duplicates().reset_index(drop=True)
print(f"Kích thước sau khi loại bỏ trùng lặp: {model_data.shape}")
print(model_data.columns.tolist())

Kích thước sau khi loại bỏ trùng lặp: (95002, 22)
['employee_id', 'age', 'gender', 'job_role', 'seniority_level', 'years_experience', 'work_mode', 'salary_usd', 'work_hours_per_week', 'sleep_hours_per_night', 'uses_therapy', 'ai_tools_daily', 'deadline_pressure_score', 'stress_score', 'burnout_score', 'phq9_score', 'phq9_category', 'gad7_score', 'gad7_category', 'burnout_level', 'seeks_mental_health_support', 'job_change_intention']


In [72]:
# Tiền xử lý 2: Loại bỏ dòng thiếu giá trị target
missing_before = model_data.isna().sum().sum()
model_data = model_data.dropna(subset=["burnout_level"]).copy()
print(f"Số giá trị thiếu trước xử lý: {missing_before}")
print(f"Kích thước sau khi xử lý target: {model_data.shape}")

Số giá trị thiếu trước xử lý: 0
Kích thước sau khi xử lý target: (95002, 22)


In [73]:
# Tiền xử lý 3: Tách target, loại bỏ ID và xóa dòng thiếu feature
TARGET = "burnout_level"
LEAKAGE_COLUMNS = ["employee_id", "burnout_score", "phq9_category", "gad7_category"]

X = model_data.drop(columns=[TARGET] + LEAKAGE_COLUMNS)
y = model_data[TARGET]

valid_rows = X.notna().all(axis=1)
removed_rows = int((~valid_rows).sum())
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

print(f"Đã xóa {removed_rows} dòng có feature thiếu")
print(f"Số feature trước mã hóa: {X.shape[1]}")
print(y.value_counts())

Đã xóa 0 dòng có feature thiếu
Số feature trước mã hóa: 17
burnout_level
Severe      27137
Moderate    25000
Low         24571
High        18294
Name: count, dtype: int64


In [74]:
# Tiền xử lý 4: Chia train/test có phân tầng theo target
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (76001, 17) | Test: (19001, 17)


In [75]:
# Tiền xử lý 5: One-hot encode và tùy chọn chuẩn hóa biến số
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

USE_STANDARDIZATION = False

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

numeric_pipeline = (
    Pipeline([("scaler", StandardScaler())])
    if USE_STANDARDIZATION
    else "passthrough"
)
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
print(f"Biến số: {len(numeric_features)} | Biến phân loại: {len(categorical_features)}")
print(f"Chuẩn hóa StandardScaler: {'Bật' if USE_STANDARDIZATION else 'Tắt'}")

Biến số: 13 | Biến phân loại: 4
Chuẩn hóa StandardScaler: Tắt


In [76]:
# Tiền xử lý 6: Fit pipeline trên train và biến đổi train/test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

burnout_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3,
}

all_levels = set(y_train.unique()) | set(y_test.unique())
unknown_levels = all_levels - set(burnout_mapping)
if unknown_levels:
    raise ValueError(f"Mức burnout chưa có trong mapping: {unknown_levels}")

y_train_encoded = y_train.map(burnout_mapping).to_numpy()
y_test_encoded = y_test.map(burnout_mapping).to_numpy()
processed_feature_names = preprocessor.get_feature_names_out()

print(f"Train sau xử lý: {X_train_processed.shape}")
print(f"Test sau xử lý: {X_test_processed.shape}")
print(f"Mapping target: {burnout_mapping}")

Train sau xử lý: (76001, 36)
Test sau xử lý: (19001, 36)
Mapping target: {'Low': 0, 'Moderate': 1, 'High': 2, 'Severe': 3}


In [77]:
# Xuất toàn bộ dữ liệu sau tiền xử lý ra một CSV duy nhất
file_suffix = "standardized_filtered" if USE_STANDARDIZATION else "preprocessed_filtered"
train_export = pd.DataFrame(X_train_processed, columns=processed_feature_names)
test_export = pd.DataFrame(X_test_processed, columns=processed_feature_names)

train_export[TARGET] = y_train.reset_index(drop=True)
test_export[TARGET] = y_test.reset_index(drop=True)
train_export[f"{TARGET}_encoded"] = y_train_encoded
test_export[f"{TARGET}_encoded"] = y_test_encoded

combined_export = pd.concat([train_export, test_export], ignore_index=True)
combined_path = f"{file_suffix}.csv"
combined_export.to_csv(combined_path, index=False)

print(f"Đã xuất: {combined_path} - {combined_export.shape}")

Đã xuất: preprocessed_filtered.csv - (95002, 38)
